# 02 · Segmentación de agua con machine learning por píxel

Clasificación supervisada por píxel: cada píxel de la imagen de radar se convierte en **una fila de características**, la
máscara del mismo día da su **etiqueta**, se entrena un clasificador y las predicciones se **vuelven a poner en su posición**
para reconstruir la máscara. Al final se compara con la U-Net del notebook 01.

1. Datos · 2. Preprocesamiento · 3. Normalización · 4. Modelos · 5. Resultados · 6. Aplicación externa (Lima 2026) · 7. Conclusiones

Requiere haber ejecutado `01_segmentacion_deep_learning.ipynb` (escenas procesadas, máscaras, validación de los datos y U-Net).

## 1. Datos

Las escenas que pasaron la validación de datos del notebook 01 (sección 2.5), con su rol.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, rasterio
from scipy import ndimage

datos = pd.read_csv("Data/procesado/validacion_datos.csv")
datos = datos[datos.sirve]
datos.groupby("rol").clave.apply(list)

## 2. Preprocesamiento: de imagen a tabla de píxeles

Las escenas ya están calibradas, filtradas y en dB (notebook 01, sección 2). Cada píxel pasa a ser una fila con 9 columnas:

| Característica | Por qué |
|---|---|
| VV, VH | el agua lisa sale oscura |
| VV − VH | relación entre polarizaciones (en dB es una resta) |
| media 3×3 y 7×7 de VV y VH | contexto del vecindario y menos ruido |
| desviación 7×7 de VV y VH | textura: el agua es homogénea, la ciudad y los cerros no |

In [ ]:
def leer(clave, carpeta="escenas"):
    with rasterio.open(f"Data/procesado/{carpeta}/{clave}.tif") as f:
        vh, vv, sombra = f.read().astype("float32")
        perfil = f.profile
    valido = (vv > 0) & (vh > 0)
    vv[~valido], vh[~valido] = np.nan, np.nan
    return 10 * np.log10(vv), 10 * np.log10(vh), valido, (sombra > 0) & valido, perfil


def etiqueta(clave):
    with rasterio.open(f"Data/procesado/etiquetas/{clave}.tif") as f:
        return f.read(1)


nombres = ["VV", "VH", "VV-VH", "VV media 3x3", "VH media 3x3", "VV media 7x7", "VH media 7x7", "VV desv 7x7", "VH desv 7x7"]


def caracteristicas(vv, vh):
    vv, vh = np.clip(np.nan_to_num(vv, nan=-30), -30, 5), np.clip(np.nan_to_num(vh, nan=-30), -30, 5)
    media = ndimage.uniform_filter
    desv = lambda x: np.sqrt(np.maximum(media(x * x, 7) - media(x, 7) ** 2, 0))
    capas = [vv, vh, vv - vh, media(vv, 3), media(vh, 3), media(vv, 7), media(vh, 7), desv(vv), desv(vh)]
    return np.stack(capas, axis=-1).reshape(-1, 9)

De cada escena de train se toman 50 000 píxeles de agua y 50 000 de tierra; de cada escena de validación, 300 000
píxeles al azar (con la proporción real de agua). `pixel` guarda la posición de cada fila en la imagen.

In [ ]:
rng = np.random.default_rng(0)
train, val = [], []
for e in datos[datos.rol != "externa"].itertuples():
    vv, vh, valido, sombra, perfil = leer(e.clave)
    et = etiqueta(e.clave).ravel()
    if e.rol == "train":
        idx = np.concatenate([rng.choice(np.flatnonzero(et == 1), 50_000), rng.choice(np.flatnonzero(et == 0), 50_000)])
    else:
        idx = rng.choice(np.flatnonzero(et != 255), 300_000)
    tabla = pd.DataFrame(caracteristicas(vv, vh)[idx], columns=nombres).assign(agua=et[idx] == 1, clave=e.clave, pixel=idx)
    (train if e.rol == "train" else val).append(tabla)
train, val = pd.concat(train, ignore_index=True), pd.concat(val, ignore_index=True)
print(f"train {len(train):,} filas ({train.agua.mean():.0%} agua) · validación {len(val):,} filas ({val.agua.mean():.1%} agua)")
train.head()

## 3. Normalización

Cada característica se estandariza (media 0, desviación 1) con un `StandardScaler` ajustado **solo con train**. Va dentro del
mismo `Pipeline` que el clasificador, así que validación, Lima 2026 o una escena nueva reciben la misma transformación.

In [ ]:
from sklearn.preprocessing import StandardScaler

escala = StandardScaler().fit(train[nombres])
pd.DataFrame({"media train": escala.mean_, "desviación train": escala.scale_,
              "media val escalada": escala.transform(val[nombres]).mean(0),
              "desv. val escalada": escala.transform(val[nombres]).std(0)}, index=nombres).round(3)

## 4. Modelos

| Modelo | Idea |
|---|---|
| Regresión logística | frontera lineal entre agua y tierra |
| Random Forest | 150 árboles con muestras al azar; vota la mayoría |
| Gradient Boosting | árboles en serie; cada uno corrige el error del anterior |

El umbral de decisión de cada uno se elige maximizando el IoU en validación.

In [ ]:
import joblib, time
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier


def metricas(pred, ref):
    tp, fp, fn = (pred & ref).sum(), (pred & ~ref).sum(), (~pred & ref).sum()
    return {"IoU": tp / max(tp + fp + fn, 1), "F1": 2 * tp / max(2 * tp + fp + fn, 1),
            "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1)}


clasificadores = {"regresion_logistica": LogisticRegression(max_iter=1000),
                  "random_forest": RandomForestClassifier(150, max_depth=16, min_samples_leaf=20, n_jobs=-1, random_state=0),
                  "gradient_boosting": HistGradientBoostingClassifier(max_iter=300, random_state=0)}
Path("Data/procesado/ml").mkdir(parents=True, exist_ok=True)
modelos, prob, umbral = {}, {}, {}
for nombre, clf in clasificadores.items():
    inicio = time.time()
    modelos[nombre] = make_pipeline(StandardScaler(), clf).fit(train[nombres], train.agua)
    prob[nombre] = modelos[nombre].predict_proba(val[nombres])[:, 1]
    umbral[nombre] = max(np.arange(0.1, 0.95, 0.05).round(2), key=lambda u: metricas(prob[nombre] >= u, val.agua.to_numpy())["IoU"])
    joblib.dump({"modelo": modelos[nombre], "umbral": umbral[nombre], "caracteristicas": nombres}, f"Data/procesado/ml/{nombre}.joblib")
    print(f"{nombre}: {time.time() - inicio:.0f} s · umbral {umbral[nombre]}")

## 5. Resultados en validación

Todos los modelos se miden sobre **los mismos píxeles de validación**. La U-Net del notebook 01 se aplica a las mismas
escenas y se leen sus probabilidades en esos mismos píxeles.

In [ ]:
import torch, segmentation_models_pytorch as smp

gpu = "cuda" if torch.cuda.is_available() else "cpu"
encoder = pd.read_csv("Data/procesado/dl/resultados_validacion.csv").encoder[0]
guardado = torch.load(f"Data/procesado/dl/unet_{encoder}.pt")
unet = smp.Unet(encoder, encoder_weights=None, in_channels=2, classes=1).to(gpu)
unet.load_state_dict(guardado["pesos"])
media, desv = np.array(guardado["media"], "float32"), np.array(guardado["desv"], "float32")


def normalizar(x):
    return np.nan_to_num((np.clip(x.astype("float32"), -30, 5) - media[:, None, None]) / desv[:, None, None])


def mapa(modelo, vv, vh):
    modelo.eval()
    x = normalizar(np.stack([vv, vh]))
    alto, ancho = vv.shape
    suma, veces = np.zeros((alto, ancho), "float32"), np.zeros((alto, ancho), "float32")
    filas = list(range(0, alto - 256, 128)) + [alto - 256]
    columnas = list(range(0, ancho - 256, 128)) + [ancho - 256]
    with torch.no_grad(), torch.autocast(gpu, dtype=torch.bfloat16):
        for i in filas:
            lote = torch.tensor(np.stack([x[:, i:i + 256, j:j + 256] for j in columnas]), dtype=torch.float32, device=gpu)
            prob = torch.sigmoid(modelo(lote)).float().cpu().numpy()[:, 0]
            for j, pj in zip(columnas, prob):
                suma[i:i + 256, j:j + 256] += pj
                veces[i:i + 256, j:j + 256] += 1
    return suma / veces


nombre_unet = f"unet_{encoder}"
prob[nombre_unet] = np.concatenate([mapa(unet, *leer(c)[:2]).ravel()[g.pixel.to_numpy()] for c, g in val.groupby("clave", sort=False)])
umbral[nombre_unet] = guardado["umbral"]
comparacion = pd.DataFrame([{"modelo": n, "familia": "U-Net" if n.startswith("unet") else "ML por píxel", "umbral": umbral[n],
                             **metricas(p >= umbral[n], val.agua.to_numpy())} for n, p in prob.items()])
comparacion = comparacion.sort_values("IoU", ascending=False)
comparacion.to_csv("Data/procesado/ml/comparacion_validacion.csv", index=False)
mejor = comparacion[comparacion.familia == "ML por píxel"].modelo.iloc[0]
print("mejor modelo por píxel:", mejor)
comparacion.round(3)

In [ ]:
from sklearn.inspection import permutation_importance

fig, ejes = plt.subplots(1, 2, figsize=(15, 4))
comparacion.set_index("modelo")[["IoU", "F1", "Precision", "Recall"]].plot.bar(ax=ejes[0], rot=20, ylim=(0, 1))
ejes[0].set_title("validación (mismos píxeles)")
muestra = val.sample(20_000, random_state=0)
for nombre, m in modelos.items():
    imp = permutation_importance(m, muestra[nombres], muestra.agua, scoring="balanced_accuracy", n_repeats=3, random_state=0)
    ejes[1].plot(imp.importances_mean, nombres, "o-", label=nombre)
ejes[1].axvline(0, color="k", lw=0.5)
ejes[1].set_title("importancia por permutación (caída de exactitud balanceada)")
ejes[1].legend()
plt.tight_layout()

### 5.1 De píxeles a filas y de vuelta a la máscara

Una ventana de 12 × 12 píxeles de validación con agua y tierra: cada píxel es una fila de la tabla, el modelo da una
probabilidad por fila y con la posición de cada fila se reconstruye la máscara.

In [ ]:
clave = val.clave.iloc[0]
vv, vh, valido, sombra, perfil = leer(clave)
et = etiqueta(clave)
fraccion = ndimage.uniform_filter((et == 1).astype("float32"), 12)
completa = ndimage.uniform_filter((et != 255).astype("float32"), 12) == 1
i, j = np.unravel_index(np.argmin(np.where(completa, abs(fraccion - 0.4), 9)), et.shape)
i, j = i - 6, j - 6
filas = caracteristicas(vv[i:i + 12, j:j + 12], vh[i:i + 12, j:j + 12])
p = modelos[mejor].predict_proba(pd.DataFrame(filas, columns=nombres))[:, 1]
display(pd.DataFrame(filas[:5], columns=nombres).assign(etiqueta=et[i:i + 12, j:j + 12].ravel()[:5], probabilidad=p[:5]).round(2))
fig, ejes = plt.subplots(1, 4, figsize=(13, 3.4))
ejes[0].imshow(vh[i:i + 12, j:j + 12], cmap="gray", vmin=-30, vmax=-5)
ejes[0].set_title("VH (dB)")
ejes[1].imshow(et[i:i + 12, j:j + 12] == 1, cmap="Blues")
ejes[1].set_title("etiqueta")
ejes[2].imshow(p.reshape(12, 12), cmap="magma", vmin=0, vmax=1)
ejes[2].set_title("probabilidad de cada fila, recolocada")
ejes[3].imshow(p.reshape(12, 12) >= umbral[mejor], cmap="Blues")
ejes[3].set_title("máscara reconstruida")
for ax in ejes:
    ax.set_xticks([])
    ax.set_yticks([])

## 6. Aplicación externa: lluvias de Lima 2026

El mejor modelo por píxel se aplica a cada escena completa, se compara con la máscara del mismo día y con la U-Net, y se
guarda la probabilidad como GeoTIFF en `Data/procesado/ml/predicciones/`.

In [ ]:
Path("Data/procesado/ml/predicciones").mkdir(exist_ok=True)
filas = []
for e in datos[datos.rol == "externa"].itertuples():
    vv, vh, valido, sombra, perfil = leer(e.clave)
    p = np.full(vv.size, np.nan, "float32")
    p[valido.ravel()] = modelos[mejor].predict_proba(pd.DataFrame(caracteristicas(vv, vh)[valido.ravel()], columns=nombres))[:, 1]
    p = p.reshape(vv.shape)
    et = etiqueta(e.clave)
    with rasterio.open(f"Data/procesado/dl/predicciones/{e.clave}.tif") as f:
        p_unet = f.read(1) / 100
    filas.append({"clave": e.clave, "agua_mascara_km2": (et == 1).sum() / 1e4, "agua_ml_km2": (p >= umbral[mejor]).sum() / 1e4,
                  "agua_unet_km2": ((p_unet >= umbral[nombre_unet]) & valido).sum() / 1e4,
                  **metricas((p >= umbral[mejor])[et != 255], (et == 1)[et != 255])})
    perfil.update(count=1, dtype="uint8", nodata=255, compress="deflate")
    with rasterio.open(f"Data/procesado/ml/predicciones/{e.clave}.tif", "w", **perfil) as f:
        f.write(np.where(valido, np.round(np.nan_to_num(p) * 100), 255).astype("uint8"), 1)
externa = pd.DataFrame(filas)
externa.to_csv("Data/procesado/ml/resultados_externa.csv", index=False)
externa.round(3)

### 6.1 Tiempo de inferencia

Mismo recorte de 1024 × 1024 píxeles (10 km²) de Lima: en ML incluye calcular las 9 características; en la U-Net, la ventana
deslizante. Mediana de 3 repeticiones.

In [ ]:
vv, vh = [a[1000:2024, 1000:2024] for a in leer(datos[datos.rol == "externa"].clave.iloc[0])[:2]]


def cronometrar(f):
    f()
    tiempos = []
    for _ in range(3):
        inicio = time.perf_counter()
        f()
        tiempos.append(time.perf_counter() - inicio)
    return np.median(tiempos)


tiempos = {n: cronometrar(lambda: m.predict_proba(pd.DataFrame(caracteristicas(vv, vh), columns=nombres))) for n, m in modelos.items()}
tiempos[nombre_unet] = cronometrar(lambda: mapa(unet, vv, vh))
pd.Series(tiempos, name="segundos").round(3)

## 7. Conclusiones

* **Mismos datos, dos familias.** ML por píxel y U-Net usan las mismas escenas, máscaras y partición, y se miden sobre los
  mismos píxeles de validación. La diferencia es cuánto ven: el ML por píxel solo sus 9 características (un vecindario de
  7 × 7), la U-Net un parche de 256 × 256 con contexto.
* **Qué esperar.** La etiqueta sale de umbrales sobre VV y VH, así que un clasificador por píxel puede reproducirla muy bien
  en validación. Donde el contexto importa es en los días con mar agitado por el viento (pasadas de la tarde, 18:33): ahí el
  mar no sale tan oscuro en VV y un modelo que solo mira el píxel lo confunde con tierra. La sección 6 lo muestra por fecha.
* **Costo.** El ML por píxel corre en CPU y no necesita GPU; la U-Net necesita GPU para entrenar pero predice más rápido una
  escena completa (sección 6.1).
* **Limitaciones comunes.** La etiqueta es un umbral del mismo día (no verdad de campo) y el agua entre edificios no se ve en
  el radar (doble rebote pared-agua). Ver la sección 7 del notebook 01.